<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/ray_tune_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ray Tune Hyperparameter Sweep

Run hyperparameter tuning sweeps directly on a Colab GPU using [Ray Tune](https://docs.ray.io/en/latest/tune/index.html).

**Key features:**
- **ASHA scheduler** for early stopping of underperforming trials
- **Single-stage sweeps** — tune one curriculum stage at a time
- **Google Drive persistence** — all results, models, and checkpoints saved to Drive
- **Warm-start support** — load a checkpoint from a prior stage or prior sweep
- **Resumable sweeps** — if your session terminates mid-sweep, completed trials are kept and partial trials restart from scratch

**Recommended runtime:** Colab Pro+ with A100 GPU (more CPU cores for MuJoCo vectorized envs)

**GPU-specific settings:**
| Setting | A100 (40GB) | L4 (24GB) | T4 (16GB) |
|---|---|---|---|
| `MAX_CONCURRENT` | 3 | 2 | 1 |
| `N_ENVS` | 8 | 4 | 4 |
| `NUM_TRIALS` | 20 | 16 | 12 |
| `TIMESTEPS_PER_TRIAL` | 4M | 3M | 2M |
| `ppo_batch_size` choices | 64–512 | 64–256 | 64–128 |
| `ppo_n_steps` choices | 1024–4096 | 1024–2048 | 1024–2048 |
| `ppo_net_arch` | all 6 | drop deep variants | small/medium/tapered |
| SAC `MAX_CONCURRENT` | 2 | 1 | 1 |
| Estimated sweep time | ~4–6h | ~6–8h | ~8–12h |

**How it compares to Vertex AI sweep:**
| | Vertex AI | Ray Tune (this notebook) |
|---|---|---|
| Scheduling | Bayesian | ASHA early stopping |
| Cost | GCP billing per trial | Included in Colab subscription |
| Parallelism | Multiple cloud workers | Concurrent trials on single GPU |
| Setup | Docker image + GCP project | Just run the notebook |

**Resuming an interrupted sweep:**
1. Re-run Sections 1–4 (Setup, Configuration, Species, Search Space)
2. In Section 2, set `RESUME = True` (auto-detects the latest sweep, or set `RESUME_SWEEP_DIR` explicitly)
3. Run Sections 5–6 — completed trials are skipped, partial/errored trials restart from scratch with the same hyperparameters

> **Why not resume mid-training?** SB3 callback state (warmup/ramp progress, EvalCallback
> best-model tracking) cannot be reliably restored from a checkpoint. Restarting partial
> trials ensures every trial runs the full warmup/ramp schedule and produces comparable results.

## 1. Setup & Installation

In [ ]:
import importlib
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Configure headless rendering for MuJoCo (must happen before mujoco import)
    os.environ["MUJOCO_GL"] = "egl"
    NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

    # Install packages only if not already present
    if importlib.util.find_spec("mujoco") is None:
        get_ipython().system(
            'pip install -q mujoco>=3.0.0 gymnasium>=0.29.0 "stable-baselines3[extra]>=2.2.0" mediapy matplotlib'
        )
    if importlib.util.find_spec("ray") is None:
        get_ipython().system('pip install -q "ray[tune]>=2.9.0"')

    import pathlib
    import subprocess

    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if importlib.util.find_spec("environments") is None:
        get_ipython().system("pip install -q -e /content/mesozoic-labs")
    print("Colab setup complete (EGL rendering enabled).")
else:
    print("Running locally — ensure ray[tune], mujoco, stable-baselines3 are installed.")

In [ ]:
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np

if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = Path("..").resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import mujoco

from environments.shared.config import load_all_stages
from environments.shared.scripts.sweep.constants import NET_ARCH_PRESETS
from environments.shared.train_base import SpeciesConfig

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

In [ ]:
# ===== Species & Algorithm =====
SPECIES = "velociraptor"  # @param ["velociraptor", "brachiosaurus", "trex"]
ALGORITHM = "ppo"         # @param ["ppo", "sac"]

# ===== Sweep Stage =====
STAGE = 1                 # @param {type:"integer"} — curriculum stage (1, 2, or 3)

# ===== Ray Tune Budget =====
# See GPU settings table in the intro cell. For L4: NUM_TRIALS=16, MAX_CONCURRENT=2 (PPO) or 1 (SAC).
NUM_TRIALS = 40           # @param {type:"integer"} — total trials to run
MAX_CONCURRENT = 2        # @param {type:"integer"} — concurrent trials (3 for A100, 2 for L4, 1 for T4/SAC)
TIMESTEPS_PER_TRIAL = 4_000_000  # @param {type:"integer"} — training timesteps per trial (3M for L4)

# ===== Training =====
N_ENVS = 4                # @param {type:"integer"} — parallel envs per trial
SEED = 42                 # @param {type:"integer"}
EVAL_FREQ = 50_000        # @param {type:"integer"} — how often to evaluate (also used as ASHA report interval)

# ===== Warm-start (optional) =====
# Path to a model checkpoint to load (e.g. from a prior stage).
# Leave empty to train from scratch.
LOAD_PATH = ""  # @param {type:"string"}

# ===== Resume (optional) =====
# Set to True to resume a previously interrupted sweep.
# Set RESUME_SWEEP_DIR to the Drive path of the previous sweep, or leave
# empty to auto-detect the latest sweep for this species/stage/algorithm.
RESUME = False            # @param {type:"boolean"}
RESUME_SWEEP_DIR = ""     # @param {type:"string"}

# ===== Google Drive =====
USE_GOOGLE_DRIVE = True   # @param {type:"boolean"}

print(f"Species:    {SPECIES}")
print(f"Algorithm:  {ALGORITHM.upper()}")
print(f"Stage:      {STAGE}")
print(f"Trials:     {NUM_TRIALS} ({MAX_CONCURRENT} concurrent)")
print(f"Timesteps:  {TIMESTEPS_PER_TRIAL:,} per trial")
print(f"Load path:  {LOAD_PATH or '(none — training from scratch)'}")
print(f"Resume:     {RESUME}{f' from {RESUME_SWEEP_DIR}' if RESUME and RESUME_SWEEP_DIR else ''}")

In [ ]:
# ============================================================
# Google Drive Storage
# ============================================================
if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/mesozoic-labs")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Results will persist to: {DRIVE_BASE}")
elif USE_GOOGLE_DRIVE and not IN_COLAB:
    print("Warning: USE_GOOGLE_DRIVE is True but not running in Colab. Using local storage.")
    DRIVE_BASE = repo_root / "logs"
else:
    DRIVE_BASE = repo_root / "logs"
    print(f"Using local storage: {DRIVE_BASE}")

# ── Sweep output directory (on Drive for persistence) ──────────────────
if RESUME:
    # Resume: reuse an existing sweep directory
    if RESUME_SWEEP_DIR:
        SWEEP_DIR = Path(RESUME_SWEEP_DIR)
    else:
        # Auto-detect: find the latest sweep directory for this species/stage/algorithm
        _sweep_parent = DRIVE_BASE / "ray_tune_sweeps" / SPECIES
        _prefix = f"stage{STAGE}_{ALGORITHM}_"
        _candidates = sorted(
            [d for d in _sweep_parent.iterdir() if d.is_dir() and d.name.startswith(_prefix)],
            key=lambda d: d.name,
            reverse=True,
        ) if _sweep_parent.exists() else []
        assert _candidates, (
            f"RESUME=True but no previous sweep found in {_sweep_parent} "
            f"matching '{_prefix}*'. Set RESUME_SWEEP_DIR explicitly."
        )
        SWEEP_DIR = _candidates[0]
    assert SWEEP_DIR.exists(), f"Resume sweep directory does not exist: {SWEEP_DIR}"
    _sweep_name = SWEEP_DIR.name
    print(f"Resuming sweep from: {SWEEP_DIR}")
else:
    # Fresh sweep: create a new timestamped directory
    SWEEP_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
    _sweep_name = f"stage{STAGE}_{ALGORITHM}_{SWEEP_TIMESTAMP}"
    SWEEP_DIR = DRIVE_BASE / "ray_tune_sweeps" / SPECIES / _sweep_name
    SWEEP_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Sweep output directory: {SWEEP_DIR}")

# ── Local storage for Ray Tune internals ───────────────────────────────
# Ray Tune's checkpoint validation writes fail on Google Drive FUSE mounts.
# Use a fast local path for Ray's internal storage, and sync best models
# to Drive periodically for crash resilience.
LOCAL_RAY_DIR = Path("/tmp/ray_tune_storage") / _sweep_name
LOCAL_RAY_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_TRIALS_DIR = Path("/tmp/ray_tune_trials") / _sweep_name
LOCAL_TRIALS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Local Ray storage:    {LOCAL_RAY_DIR}")
print(f"Local trials dir:     {LOCAL_TRIALS_DIR}")
print(f"Drive sync target:    {SWEEP_DIR}")

## 3. Load Species & Stage Configs

In [ ]:
import importlib

_SPECIES_MAP = {
    "velociraptor": {
        "module": "environments.velociraptor.envs.raptor_env",
        "class": "RaptorEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=strike",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["strike_success", "bite_success"],
    },
    "trex": {
        "module": "environments.trex.envs.trex_env",
        "class": "TRexEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=bite",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["bite_success", "strike_success"],
    },
    "brachiosaurus": {
        "module": "environments.brachiosaurus.envs.brachio_env",
        "class": "BrachioEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=food_reach",
        "height_label": "Torso height",
        "stage3_section_label": "Food Reaching",
        "success_keys": ["food_reached"],
    },
}

assert SPECIES in _SPECIES_MAP, f"Unknown species: {SPECIES}. Choose from: {list(_SPECIES_MAP.keys())}"
_info = _SPECIES_MAP[SPECIES]
_mod = importlib.import_module(_info["module"])
EnvClass = getattr(_mod, _info["class"])

SPECIES_CFG = SpeciesConfig(
    species=SPECIES,
    env_class=EnvClass,
    stage_descriptions=_info["stage_descriptions"],
    height_label=_info["height_label"],
    stage3_section_label=_info["stage3_section_label"],
    success_keys=_info["success_keys"],
)

STAGE_CONFIGS = load_all_stages(SPECIES)

env = EnvClass()
print(f"Environment: {EnvClass.__name__}")
print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space.shape}")
for stage_num, cfg in STAGE_CONFIGS.items():
    print(f"  Stage {stage_num}: {cfg['name']} — {cfg['description']}")
env.close()

## 4. Search Space

Translates the same search spaces used by the Vertex AI sweep into Ray Tune format.

**Per-stage rationale (same as Vertex AI notebook):**
- **Stage 1 (balance):** `alive_bonus` is the dominant reward signal — worth sweeping
- **Stage 2 (locomotion):** `alive_bonus` must stay low to avoid standing-trap
- **Stage 3 (behavior):** Only sweep algo params; `alive_bonus` is intentionally small

In [ ]:
from ray import tune

# ── Shared algorithm hyperparameters ──────────────────────────────────────────
_PPO_ALGO_SPACE = {
    "ppo_learning_rate": tune.loguniform(1e-5, 3e-4),
    "ppo_ent_coef": tune.loguniform(1e-4, 0.05),
    "ppo_batch_size": tune.choice([64, 128, 256, 512]),
    "ppo_gamma": tune.uniform(0.97, 0.999),
    "ppo_n_steps": tune.choice([1024, 2048, 4096]),
    "ppo_n_epochs": tune.choice([3, 6, 10]),
    "ppo_net_arch": tune.choice(["small", "medium", "large", "deep", "tapered", "deep_tapered"]),
}

_SAC_ALGO_SPACE = {
    "sac_learning_rate": tune.loguniform(1e-5, 3e-4),
    "sac_batch_size": tune.choice([128, 256, 512]),
    "sac_gamma": tune.uniform(0.97, 0.999),
    "sac_net_arch": tune.choice(["small", "medium", "large", "tapered", "deep_tapered"]),
}

_ALGO_SPACE = _PPO_ALGO_SPACE if ALGORITHM == "ppo" else _SAC_ALGO_SPACE

# ── Per-stage, per-species search spaces ───────────────────────────────────────
# Aligned with the Vertex AI sweep (configs/sweep_ppo.json) and extended with
# species-specific reward signals.  Stage 3 attack/goal parameters differ by
# species: velociraptor uses strike_*, trex uses bite_*, brachiosaurus uses
# food_reach_*.

# --- Stage 1: Balance ---
_STAGE1_COMMON = {
    **_ALGO_SPACE,
    "env_alive_bonus": tune.uniform(1.0, 5.0),
    "env_energy_penalty_weight": tune.loguniform(0.01, 0.1),
}

_STAGE1_BY_SPECIES = {
    "velociraptor": {
        **_STAGE1_COMMON,
        "env_posture_weight": tune.uniform(0.5, 3.0),
        "env_nosedive_weight": tune.uniform(0.5, 3.0),
        "env_drift_penalty_weight": tune.uniform(0.0, 0.5),
        "env_spin_penalty_weight": tune.uniform(0.0, 0.5),
    },
    "trex": {
        **_STAGE1_COMMON,
        "env_posture_weight": tune.uniform(0.5, 4.0),
        "env_nosedive_weight": tune.uniform(0.5, 5.0),
        "env_height_weight": tune.uniform(0.5, 2.0),
        "env_spin_penalty_weight": tune.uniform(0.0, 0.5),
        "env_smoothness_weight": tune.uniform(0.02, 0.2),
    },
    "brachiosaurus": {
        **_STAGE1_COMMON,
        "env_gait_stability_weight": tune.uniform(0.02, 0.2),
    },
}

# --- Stage 2: Locomotion ---
_STAGE2_COMMON = {
    **_ALGO_SPACE,
    "env_forward_vel_weight": tune.uniform(1.0, 4.0),
    "env_forward_vel_max": tune.uniform(2.0, 5.0),
    "env_alive_bonus": tune.uniform(0.1, 2.0),
    "env_energy_penalty_weight": tune.loguniform(0.001, 0.01),
    "env_heading_weight": tune.uniform(0.0, 0.6),
    "env_lateral_penalty_weight": tune.uniform(0.0, 0.3),
    "curriculum_warmup_timesteps": tune.choice([50000, 100000, 200000, 300000]),
    "curriculum_warmup_clip_range": tune.uniform(0.01, 0.05),
    "curriculum_warmup_ent_coef": tune.loguniform(0.005, 0.05),
    "curriculum_ramp_timesteps": tune.choice([200000, 500000, 1000000, 2000000]),
    "curriculum_ramp_start_value": tune.uniform(0.05, 0.3),
}

_STAGE2_BY_SPECIES = {
    "velociraptor": {
        **_STAGE2_COMMON,
        "env_posture_weight": tune.uniform(0.0, 0.5),
        "env_nosedive_weight": tune.uniform(0.1, 0.8),
    },
    "trex": {
        **_STAGE2_COMMON,
        "env_posture_weight": tune.uniform(0.1, 1.0),
        "env_nosedive_weight": tune.uniform(0.3, 2.0),
        "env_height_weight": tune.uniform(0.5, 2.0),
    },
    "brachiosaurus": {
        **_STAGE2_COMMON,
        "env_gait_stability_weight": tune.uniform(0.01, 0.1),
    },
}

# --- Stage 3: Attack / Goal ---
_STAGE3_CURRICULUM = {
    "curriculum_warmup_timesteps": tune.choice([50000, 100000, 200000, 300000]),
    "curriculum_warmup_clip_range": tune.uniform(0.01, 0.05),
    "curriculum_warmup_ent_coef": tune.loguniform(0.005, 0.05),
    "curriculum_ramp_timesteps": tune.choice([500000, 1000000, 2000000, 4000000]),
    "curriculum_ramp_start_value": tune.uniform(0.05, 0.3),
}

_STAGE3_COMMON = {
    **_ALGO_SPACE,
    "env_forward_vel_weight": tune.uniform(0.1, 1.0),
    "env_alive_bonus": tune.loguniform(0.01, 0.1),
    "env_posture_weight": tune.uniform(0.0, 0.3),
    "env_nosedive_weight": tune.uniform(0.1, 0.6),
    "env_heading_weight": tune.uniform(0.1, 0.6),
    **_STAGE3_CURRICULUM,
}

_STAGE3_BY_SPECIES = {
    "velociraptor": {
        **_STAGE3_COMMON,
        "env_strike_bonus": tune.loguniform(10.0, 100.0),
        "env_strike_approach_weight": tune.uniform(1.0, 5.0),
        "env_strike_proximity_weight": tune.uniform(0.1, 1.0),
        "env_strike_claw_proximity_weight": tune.uniform(0.5, 4.0),
    },
    "trex": {
        **_STAGE3_COMMON,
        "env_bite_bonus": tune.loguniform(10.0, 100.0),
        "env_bite_approach_weight": tune.uniform(1.0, 5.0),
        "env_lateral_penalty_weight": tune.uniform(0.1, 0.5),
    },
    "brachiosaurus": {
        **_ALGO_SPACE,
        "env_forward_vel_weight": tune.uniform(0.5, 2.0),
        "env_alive_bonus": tune.uniform(0.05, 0.3),
        "env_energy_penalty_weight": tune.loguniform(0.0005, 0.005),
        "env_gait_stability_weight": tune.uniform(0.005, 0.05),
        "env_food_reach_bonus": tune.loguniform(5.0, 50.0),
        "env_food_reach_threshold": tune.uniform(0.3, 0.8),
        "env_food_approach_weight": tune.uniform(0.5, 3.0),
        **_STAGE3_CURRICULUM,
    },
}

SEARCH_SPACE_PER_STAGE = {
    1: _STAGE1_BY_SPECIES.get(SPECIES, _STAGE1_BY_SPECIES["velociraptor"]),
    2: _STAGE2_BY_SPECIES.get(SPECIES, _STAGE2_BY_SPECIES["velociraptor"]),
    3: _STAGE3_BY_SPECIES.get(SPECIES, _STAGE3_BY_SPECIES["velociraptor"]),
}

SEARCH_SPACE = SEARCH_SPACE_PER_STAGE[STAGE]

print(f"Search space for Stage {STAGE} / {SPECIES} ({ALGORITHM.upper()}): {len(SEARCH_SPACE)} params")
for name in SEARCH_SPACE:
    print(f"  {name}")

## 5. Trial Training Function

Each Ray Tune trial runs this function. It:
1. Applies the sampled hyperparameters to the TOML stage config
2. Trains using the same `train()` infrastructure as the CLI and Vertex AI
3. Reports `best_mean_reward` **and a Ray checkpoint** to Ray Tune at each evaluation (for ASHA)
4. Always trains from scratch (or from a warm-start checkpoint for stages 2+) — partial trials are **not** resumed mid-training to avoid callback state corruption

In [ ]:
import copy
import logging
import shutil
import tempfile

from ray import tune
from ray.train import Checkpoint
from ray.tune import Callback
from stable_baselines3.common.callbacks import BaseCallback

# Ray Train v2 decoupled RunConfig from Tune — use the Tune-namespaced
# versions for Tuner configuration to avoid Train v2 deprecation errors.
from ray.tune import RunConfig as TuneRunConfig
from ray.tune import CheckpointConfig as TuneCheckpointConfig

logger = logging.getLogger(__name__)

# Suppress noisy tensorboardX NaN/Inf warnings and Ray experiment_state
# snapshot warnings at the module level (applies to the driver process).
logging.getLogger("tensorboardX").setLevel(logging.ERROR)


def _sync_to_drive(src_dir, drive_dir, label=""):
    """Copy files from local dir to Drive, tolerating FUSE flakiness."""
    src_dir = Path(src_dir)
    drive_dir = Path(drive_dir)
    if not src_dir.exists():
        return
    drive_dir.mkdir(parents=True, exist_ok=True)
    for src_file in src_dir.iterdir():
        if src_file.is_file():
            try:
                shutil.copy2(str(src_file), str(drive_dir / src_file.name))
            except OSError as e:
                logger.warning(f"Drive sync failed for {src_file.name}: {e}")


class RayTuneReportCallback(BaseCallback):
    """SB3 callback that reports eval metrics + checkpoints to Ray Tune.

    After each evaluation, reports metrics to ASHA and saves a Ray-native
    checkpoint containing the SB3 model and VecNormalize stats. Also syncs
    the best model to Google Drive for crash resilience.
    """

    def __init__(self, eval_callback, train_env, model_ref, algorithm, stage,
                 drive_best_model_dir=None, verbose=0):
        super().__init__(verbose)
        self.eval_callback = eval_callback
        self.train_env = train_env
        self._model_ref = model_ref  # mutable container [model] so we always get current ref
        self.algorithm = algorithm
        self.stage = stage
        self._last_eval_count = 0
        self._drive_best_model_dir = drive_best_model_dir
        self._best_mean_reward = float("-inf")

    def _on_step(self) -> bool:
        # Only report when a new evaluation has completed (avoid duplicate reports)
        current_eval_count = len(getattr(self.eval_callback, "evaluations_timesteps", []))
        if current_eval_count <= self._last_eval_count:
            return True
        if not hasattr(self.eval_callback, "last_mean_reward") or self.eval_callback.last_mean_reward is None or self.eval_callback.last_mean_reward == float("-inf"):
            return True

        self._last_eval_count = current_eval_count

        # Save a Ray checkpoint with the SB3 model + VecNormalize
        with tempfile.TemporaryDirectory() as tmpdir:
            # SB3 .save() appends .zip automatically; use consistent base name
            model_base = Path(tmpdir) / "model"
            self._model_ref[0].save(str(model_base))
            vecnorm_path = Path(tmpdir) / "vecnorm.pkl"
            self.train_env.save(str(vecnorm_path))

            checkpoint = Checkpoint.from_directory(tmpdir)
            tune.report(
                {
                    "best_mean_reward": float(self.eval_callback.best_mean_reward),
                    "last_mean_reward": float(self.eval_callback.last_mean_reward),
                    "timesteps": self.num_timesteps,
                },
                checkpoint=checkpoint,
            )

        # Sync best model to Drive whenever EvalCallback finds a new best
        if (self._drive_best_model_dir
                and self.eval_callback.best_mean_reward > self._best_mean_reward):
            self._best_mean_reward = self.eval_callback.best_mean_reward
            best_src = Path(self.eval_callback.best_model_save_path)
            _sync_to_drive(best_src, self._drive_best_model_dir,
                           label=f"best@{self.num_timesteps}")

        return True


def _apply_hpt_config_to_stage(stage_configs, stage, hpt_config, algorithm):
    """Apply Ray Tune sampled hyperparameters to the stage config dict.

    Uses the same naming convention as the Vertex AI sweep:
    - ppo_* / sac_*       -> stage_configs[stage]["ppo_kwargs"] / ["sac_kwargs"]
    - env_*               -> stage_configs[stage]["env_kwargs"]
    - curriculum_*        -> stage_configs[stage]["curriculum_kwargs"]
    - *_net_arch          -> policy_kwargs.net_arch (resolved via NET_ARCH_PRESETS)
    """
    config = stage_configs[stage]
    algo_key = f"{algorithm}_kwargs"

    for key, value in hpt_config.items():
        for prefix in ("ppo", "sac", "env", "curriculum"):
            if key.startswith(prefix + "_"):
                param = key[len(prefix) + 1:]
                if prefix in ("ppo", "sac"):
                    if param == "net_arch":
                        config[algo_key].setdefault("policy_kwargs", {})["net_arch"] = NET_ARCH_PRESETS[value]
                    else:
                        # Cast discrete params to int
                        if param in ("batch_size", "n_steps", "n_epochs"):
                            value = int(value)
                        config[algo_key][param] = value
                elif prefix == "env":
                    config["env_kwargs"][param] = value
                elif prefix == "curriculum":
                    if param in ("warmup_timesteps", "ramp_timesteps"):
                        value = int(value)
                    config["curriculum_kwargs"][param] = value
                break


def train_trial(config):
    """Ray Tune trainable function for a single hyperparameter trial.

    This function runs inside a Ray worker. It reuses the project's existing
    training infrastructure (create_vec_env, SB3 model creation, callbacks)
    rather than calling train() directly, because we need to inject the
    RayTuneReportCallback to report intermediate metrics to ASHA.

    Each trial always trains from scratch (or from a warm-start checkpoint
    for stages 2+). Mid-training resume is intentionally not supported because
    callback state (warmup/ramp progress, EvalCallback best tracking) cannot
    be reliably restored. If a sweep is interrupted, partial trials are
    restarted with the same hyperparameters.
    """
    import os
    os.environ["MUJOCO_GL"] = "egl"
    # Suppress noisy warnings inside Ray worker processes
    os.environ["TUNE_WARN_EXCESSIVE_EXPERIMENT_CHECKPOINT_SYNC_THRESHOLD_S"] = "0"
    import logging as _logging
    _logging.getLogger("tensorboardX").setLevel(_logging.ERROR)
    _logging.getLogger("ray.tune.experiment_state").setLevel(_logging.ERROR)

    from stable_baselines3 import PPO, SAC
    from stable_baselines3.common.callbacks import (
        CallbackList,
        CheckpointCallback,
        EvalCallback,
    )

    from environments.shared.curriculum import (
        EvalCollapseEarlyStopCallback,
        RewardRampCallback,
        SaveVecNormalizeCallback,
        StageWarmupCallback,
        load_vecnorm_stats,
    )
    from environments.shared.train_base import (
        create_vec_env,
        linear_schedule,
        cosine_schedule,
    )

    # Unpack fixed params from config
    species = config["_species"]
    algorithm = config["_algorithm"]
    stage = config["_stage"]
    timesteps = config["_timesteps"]
    n_envs = config["_n_envs"]
    seed = config["_seed"]
    eval_freq = config["_eval_freq"]
    load_path = config.get("_load_path") or None
    local_trials_dir = config.get("_local_trials_dir")
    drive_sweep_dir = config.get("_drive_sweep_dir")

    # Load species config
    if species == "velociraptor":
        from environments.velociraptor.scripts.train_sb3 import SPECIES_CONFIG
    elif species == "brachiosaurus":
        from environments.brachiosaurus.scripts.train_sb3 import SPECIES_CONFIG
    elif species == "trex":
        from environments.trex.scripts.train_sb3 import SPECIES_CONFIG

    stage_configs = load_all_stages(species)

    # Apply sampled hyperparameters (skip keys starting with _ which are fixed params)
    hpt_params = {k: v for k, v in config.items() if not k.startswith("_")}
    _apply_hpt_config_to_stage(stage_configs, stage, hpt_params, algorithm)

    stage_config = stage_configs[stage]

    # Setup output directory — use local storage for speed and reliability,
    # sync best models to Drive for crash resilience.
    trial_id = tune.get_context().get_trial_id() or "local"
    trial_dir = Path(local_trials_dir) / trial_id if local_trials_dir else Path(f"/tmp/ray_tune_trial_{trial_id}")
    trial_dir.mkdir(parents=True, exist_ok=True)
    model_dir = trial_dir / "models"
    model_dir.mkdir(exist_ok=True)

    # Drive directory for this trial's best model (synced periodically)
    drive_trial_dir = None
    drive_best_model_dir = None
    if drive_sweep_dir:
        drive_trial_dir = Path(drive_sweep_dir) / "trials" / trial_id
        drive_best_model_dir = drive_trial_dir / "models"

    # Create environments
    train_env = create_vec_env(SPECIES_CONFIG, stage_configs, stage, n_envs, seed)
    eval_env = create_vec_env(SPECIES_CONFIG, stage_configs, stage, 1, seed + 1000, use_subproc=False)

    try:
        # Create or load model
        alg_cls = SAC if algorithm == "sac" else PPO
        algo_key = "sac_kwargs" if algorithm == "sac" else "ppo_kwargs"
        alg_kwargs = stage_config[algo_key].copy()
        alg_kwargs["verbose"] = 0
        alg_kwargs["tensorboard_log"] = str(trial_dir / "tensorboard")

        if algorithm == "ppo":
            lr_end = alg_kwargs.pop("learning_rate_end", None)
            lr_schedule_type = alg_kwargs.pop("lr_schedule", "linear")
            if lr_end is not None:
                lr_start = alg_kwargs["learning_rate"]
                if lr_schedule_type == "cosine":
                    alg_kwargs["learning_rate"] = cosine_schedule(lr_start, lr_end)
                else:
                    alg_kwargs["learning_rate"] = linear_schedule(lr_start, lr_end)

            clip_range_end = alg_kwargs.pop("clip_range_end", None)
            if clip_range_end is not None:
                clip_start = alg_kwargs["clip_range"]
                alg_kwargs["clip_range"] = linear_schedule(clip_start, clip_range_end)

        policy_kwargs = alg_kwargs.pop("policy_kwargs", None)

        if load_path:
            # ── Warm-start from a prior stage model ───────────────────────────
            vecnorm_path = load_path.replace(".zip", "") + "_vecnorm.pkl"
            if not vecnorm_path.endswith("_vecnorm.pkl"):
                vecnorm_path = load_path + "_vecnorm.pkl"
            if not load_vecnorm_stats(vecnorm_path, train_env, eval_env):
                eval_env.training = False
                eval_env.norm_reward = False
            model = alg_cls.load(load_path, env=train_env, **alg_kwargs)
        else:
            # ── Fresh training ────────────────────────────────────────────────
            eval_env.training = False
            eval_env.norm_reward = False
            model = alg_cls("MlpPolicy", train_env, policy_kwargs=policy_kwargs, **alg_kwargs)

        # Callbacks
        callbacks = []

        save_vecnorm_cb = SaveVecNormalizeCallback(
            save_path=str(model_dir / "best_model_vecnorm.pkl"),
        )
        eval_callback = EvalCallback(
            eval_env,
            best_model_save_path=str(model_dir),
            log_path=str(trial_dir),
            eval_freq=eval_freq // n_envs,
            n_eval_episodes=30,
            deterministic=True,
            render=False,
            verbose=0,
            callback_on_new_best=save_vecnorm_cb,
        )
        callbacks.append(eval_callback)

        # Report metrics + Ray checkpoint after each eval, sync best to Drive
        model_ref = [model]  # mutable container so callback always has current model
        callbacks.append(RayTuneReportCallback(
            eval_callback, train_env, model_ref, algorithm, stage,
            drive_best_model_dir=drive_best_model_dir,
        ))

        # SB3 checkpoint callback (for post-sweep analysis only — Ray-level
        # checkpoints from RayTuneReportCallback are used for ASHA decisions).
        # Save at 5x the eval interval to reduce IO and avoid triggering
        # excessive experiment state snapshots from num_to_keep enforcement.
        callbacks.append(CheckpointCallback(
            save_freq=max(5 * eval_freq // n_envs, 1),
            save_path=str(model_dir),
            name_prefix=f"stage{stage}",
            save_vecnormalize=True,
        ))

        # Early stop on reward collapse
        callbacks.append(EvalCollapseEarlyStopCallback(eval_callback=eval_callback, verbose=0))

        # Stage transition callbacks (stages 2+)
        cur_kwargs = stage_config.get("curriculum_kwargs", {})
        if stage > 1 and load_path:
            if algorithm == "ppo":
                callbacks.append(StageWarmupCallback(
                    warmup_timesteps=cur_kwargs.get("warmup_timesteps", 100_000),
                    warmup_clip_range=cur_kwargs.get("warmup_clip_range", 0.02),
                    warmup_ent_coef=cur_kwargs.get("warmup_ent_coef", 0.02),
                ))
            target_fwd_weight = stage_config["env_kwargs"].get("forward_vel_weight", 1.0)
            callbacks.append(RewardRampCallback(
                attr_name="forward_vel_weight",
                start_value=cur_kwargs.get("ramp_start_value", 0.1),
                end_value=target_fwd_weight,
                ramp_timesteps=cur_kwargs.get("ramp_timesteps", 500_000),
            ))

        # Train
        model.learn(
            total_timesteps=timesteps,
            callback=CallbackList(callbacks),
            progress_bar=False,
        )

        # Save final model locally
        final_path = model_dir / f"stage{stage}_final"
        model.save(str(final_path))
        train_env.save(str(final_path) + "_vecnorm.pkl")

        # Sync final model + best model to Drive
        if drive_best_model_dir:
            _sync_to_drive(model_dir, drive_best_model_dir, label="final")

        # Final report with checkpoint for post-sweep analysis
        with tempfile.TemporaryDirectory() as tmpdir:
            model.save(str(Path(tmpdir) / "model"))
            train_env.save(str(Path(tmpdir) / "vecnorm.pkl"))
            checkpoint = Checkpoint.from_directory(tmpdir)
            tune.report(
                {
                    "best_mean_reward": float(eval_callback.best_mean_reward),
                    "timesteps": timesteps,
                    "done": True,
                },
                checkpoint=checkpoint,
            )
    finally:
        train_env.close()
        eval_env.close()


print("Trial function defined.")

## 6. Run the Sweep

Ray Tune manages the trial scheduling. The **ASHA scheduler** stops underperforming
trials early based on intermediate `best_mean_reward` reports, saving significant compute.

**ASHA parameters:**
- `grace_period`: Minimum number of reports before a trial can be stopped (default: 20 = ~1M timesteps)
- `reduction_factor`: At each rung, keep the top 1/factor trials (default: 3 = keep top ~33%)
- `max_t`: Maximum reports per trial (auto-calculated from timesteps / eval_freq)

**Resuming:** When `RESUME = True`, `Tuner.restore()` reloads the experiment state from
Google Drive. Completed trials are skipped. Partial/errored trials restart from scratch
with the same hyperparameters — this ensures the full warmup/ramp schedule runs and
`EvalCallback` tracking starts fresh, producing results comparable to uninterrupted trials.

In [ ]:
import os
import time

import ray
from ray import tune
from ray.tune import Callback
from ray.tune.schedulers import ASHAScheduler

# Suppress noisy Ray experiment_state snapshot warnings
os.environ["TUNE_WARN_EXCESSIVE_EXPERIMENT_CHECKPOINT_SYNC_THRESHOLD_S"] = "0"

# Shutdown any existing Ray instance
if ray.is_initialized():
    ray.shutdown()

# Initialize Ray — let it auto-detect resources
ray.init(ignore_reinit_error=True)
print(f"Ray initialized: {ray.cluster_resources()}")

# ASHA scheduler for early stopping
max_reports = TIMESTEPS_PER_TRIAL // EVAL_FREQ
GRACE_PERIOD = 20            # Don't stop before 8 eval reports (~1M steps)
REDUCTION_FACTOR = 3        # Keep top ~33% of trials at each rung
scheduler = ASHAScheduler(
    metric="best_mean_reward",
    mode="max",
    max_t=max_reports,
    grace_period=GRACE_PERIOD,
    reduction_factor=REDUCTION_FACTOR,
)


class TrialTerminationCallback(Callback):
    """Print a status summary when a trial terminates or periodically.

    Replaces the old ``TrialTerminationReporter`` (a ``JupyterNotebookReporter``
    subclass).  The modern ``tune.Callback`` API is the supported way to hook
    into experiment events; ``progress_reporter`` is a legacy DeveloperAPI
    that the new Ray output engine bypasses.

    The callback prints a one-line summary on every trial completion and a
    full status table every ``report_interval_s`` seconds (default 5 min)
    whenever *any* trial reports a result.
    """

    METRIC_COLS = ("best_mean_reward", "last_mean_reward", "timesteps")

    def __init__(self, report_interval_s: int = 300):
        self._report_interval_s = report_interval_s
        self._last_report_time = 0.0

    def on_trial_complete(self, iteration, trials, trial, **info):
        """Always print when a trial finishes."""
        metrics = {k: trial.last_result.get(k) for k in self.METRIC_COLS}
        metrics_str = "  ".join(
            f"{k}={v:.2f}" if isinstance(v, float) else f"{k}={v}"
            for k, v in metrics.items()
        )
        n_done = sum(1 for t in trials if t.status == "TERMINATED")
        print(f"[Trial {trial.trial_id} DONE] ({n_done}/{len(trials)} complete)  {metrics_str}")

    def on_trial_result(self, iteration, trials, trial, result, **info):
        """Print a full status table at most every ``report_interval_s``."""
        now = time.time()
        if now - self._last_report_time < self._report_interval_s:
            return
        self._last_report_time = now

        header = (
            f"\n{'trial_id':<16}"
            + "".join(f"{c:>20}" for c in self.METRIC_COLS)
            + f"{'status':>14}"
        )
        print(header)
        print("-" * len(header))
        for t in trials:
            cols = "".join(
                f"{t.last_result.get(c, ''):>20}"
                if not isinstance(t.last_result.get(c), float)
                else f"{t.last_result[c]:>20.2f}"
                for c in self.METRIC_COLS
            )
            print(f"{t.trial_id:<16}{cols}{t.status:>14}")
        print()


trial_callback = TrialTerminationCallback(report_interval_s=300)

# Fixed parameters passed to every trial (prefixed with _ to distinguish from search space)
fixed_config = {
    "_species": SPECIES,
    "_algorithm": ALGORITHM,
    "_stage": STAGE,
    "_timesteps": TIMESTEPS_PER_TRIAL,
    "_n_envs": N_ENVS,
    "_seed": SEED,
    "_eval_freq": EVAL_FREQ,
    "_load_path": LOAD_PATH or "",
    "_local_trials_dir": str(LOCAL_TRIALS_DIR),
    "_drive_sweep_dir": str(SWEEP_DIR),
}

# Merge fixed params with the search space
full_config = {**fixed_config, **SEARCH_SPACE}

print(f"\nStarting Ray Tune sweep:")
print(f"  Species:    {SPECIES}")
print(f"  Stage:      {STAGE}")
print(f"  Algorithm:  {ALGORITHM.upper()}")
print(f"  Trials:     {NUM_TRIALS} ({MAX_CONCURRENT} concurrent)")
print(f"  Timesteps:  {TIMESTEPS_PER_TRIAL:,} per trial")
print(f"  Params:     {len(SEARCH_SPACE)} search-space dimensions")
print(f"  Scheduler:  ASHA (grace={GRACE_PERIOD}, reduction={REDUCTION_FACTOR})")
print(f"  Callback:   status table every {trial_callback._report_interval_s}s or on trial completion")
print(f"  Local:      {LOCAL_RAY_DIR}")
print(f"  Drive:      {SWEEP_DIR}")
if LOAD_PATH:
    print(f"  Warm-start: {LOAD_PATH}")

In [ ]:
# Run the sweep
# Allocate fractional GPU per trial to prevent OOM with concurrent trials
_gpu_fraction = 1.0 / max(MAX_CONCURRENT, 1)
_trainable = tune.with_resources(train_trial, {"cpu": 2, "gpu": _gpu_fraction})

if RESUME:
    # Resume from a previous sweep's Ray results directory.
    # Completed trials are kept; partial/errored trials are restarted from
    # scratch with the same hyperparameters. This avoids callback state
    # issues (warmup/ramp progress, EvalCallback best_mean_reward) that
    # make mid-training resume produce unreliable results.
    _experiment_name = f"{SPECIES}_stage{STAGE}_{ALGORITHM}"
    _experiment_dir = LOCAL_RAY_DIR / _experiment_name

    assert _experiment_dir.exists(), (
        f"Cannot resume: experiment directory not found at {_experiment_dir}. "
        f"Check that LOCAL_RAY_DIR points to the correct local storage."
    )

    print(f"Restoring Tuner from: {_experiment_dir}")
    print("  Completed trials will be kept; partial trials will restart from scratch.")
    tuner = tune.Tuner.restore(
        path=str(_experiment_dir),
        trainable=_trainable,
        resume_unfinished=True,
        resume_errored=True,
        # Don't pass checkpoints to restarted trials — train from scratch
        # so warmup/ramp callbacks run the full schedule
        param_space=full_config,
    )
else:
    tuner = tune.Tuner(
        _trainable,
        param_space=full_config,
        tune_config=tune.TuneConfig(
            scheduler=scheduler,
            num_samples=NUM_TRIALS,
            max_concurrent_trials=MAX_CONCURRENT,
        ),
        run_config=TuneRunConfig(
            name=f"{SPECIES}_stage{STAGE}_{ALGORITHM}",
            # Use local storage — Google Drive FUSE mounts fail Ray's
            # checkpoint validation writes. Best models are synced to
            # Drive by the RayTuneReportCallback for crash resilience.
            storage_path=str(LOCAL_RAY_DIR),
            checkpoint_config=TuneCheckpointConfig(
                # Keep checkpoints for post-sweep analysis, but they are not
                # used for mid-training resume (partial trials restart fresh)
                num_to_keep=2,
            ),
            # verbose=2 shows status updates + brief trial results.
            # Detailed per-trial logging is handled by TrialTerminationCallback.
            verbose=2,
            callbacks=[trial_callback],
        ),
    )

results = tuner.fit()
print("\nSweep complete!")

# ── Sync Ray Tune results to Drive for persistence ────────────────────
import shutil as _shutil

_local_experiment_dir = LOCAL_RAY_DIR / f"{SPECIES}_stage{STAGE}_{ALGORITHM}"
_drive_ray_results_dir = SWEEP_DIR / "ray_results"
if _local_experiment_dir.exists():
    print(f"Syncing Ray results to Drive: {_drive_ray_results_dir}")
    try:
        _shutil.copytree(
            str(_local_experiment_dir),
            str(_drive_ray_results_dir / _local_experiment_dir.name),
            dirs_exist_ok=True,
        )
        print("  Ray results synced to Drive successfully.")
    except OSError as e:
        print(f"  Warning: Drive sync of Ray results failed: {e}")
        print("  Trial best models were already synced individually during training.")

# ── Generate collected_results.csv ────────────────────────────────────
import pandas as _pd
from environments.shared.scripts.sweep.results import (
    plot_sweep_results,
    write_results_csv,
    _evaluate_curriculum_gate,
)

_results_df = results.get_dataframe()
if "best_mean_reward" in _results_df.columns:
    _results_df = _results_df.sort_values("best_mean_reward", ascending=False)

_sweep_rows = []
_stage_config = STAGE_CONFIGS[STAGE]
for _, _rt_row in _results_df.iterrows():
    _row = {"trial_id": str(_rt_row.get("trial_id", "")), "stage": STAGE}
    for _col in _results_df.columns:
        if _col.startswith(("ppo_", "sac_", "env_")):
            _row[_col] = _rt_row[_col]
    _row["best_mean_reward"] = _rt_row.get("best_mean_reward")
    _row["best_mean_episode_length"] = _rt_row.get("best_mean_episode_length")
    _row["last_mean_reward"] = _rt_row.get("last_mean_reward")
    _row["last_mean_episode_length"] = _rt_row.get("last_mean_episode_length")
    _row["mean_forward_vel"] = _rt_row.get("mean_forward_vel")
    _row["std_forward_vel"] = _rt_row.get("std_forward_vel")
    _row["mean_success_rate"] = _rt_row.get("mean_success_rate")
    _row["training_duration_seconds"] = _rt_row.get("training_duration_seconds")
    _cur = _stage_config.get("curriculum_kwargs", {})
    _row["reward_threshold"] = _cur.get("min_avg_reward")
    _row["ep_length_threshold"] = _cur.get("min_avg_episode_length")
    _row["forward_vel_threshold"] = _cur.get("min_avg_forward_vel")
    _row["success_rate_threshold"] = _cur.get("min_success_rate")
    _passed, _ = _evaluate_curriculum_gate(
        _row["best_mean_reward"], _row,
        _row["reward_threshold"], _row["ep_length_threshold"],
        _row["forward_vel_threshold"], _row["success_rate_threshold"],
    )
    _row["stage_passed"] = _passed
    _sweep_rows.append(_row)

_collected_csv = write_results_csv(_sweep_rows, SWEEP_DIR / "collected_results.csv")
print(f"\nCollected results CSV saved to: {_collected_csv}")

## 7. Results & Analysis

In [ ]:
import pandas as pd

# Get results as a DataFrame
results_df = results.get_dataframe()

# Sort by best reward
if "best_mean_reward" in results_df.columns:
    results_df = results_df.sort_values("best_mean_reward", ascending=False)

# Display the hyperparameter columns (filter out internal Ray columns)
param_cols = [c for c in results_df.columns if c.startswith(("ppo_", "sac_", "env_", "curriculum_"))]
metric_cols = ["best_mean_reward", "last_mean_reward", "timesteps"]
display_cols = [c for c in metric_cols + param_cols if c in results_df.columns]

print(f"\n{'=' * 60}")
print(f"Sweep Results: {SPECIES} Stage {STAGE} ({ALGORITHM.upper()})")
print(f"{'=' * 60}")
display(results_df[display_cols].head(20))

In [ ]:
# Best trial details
best_result = results.get_best_result(metric="best_mean_reward", mode="max")

print(f"\nBest Trial:")
print(f"  best_mean_reward: {best_result.metrics.get('best_mean_reward', 'N/A'):.4f}")
print(f"  Hyperparameters:")
for key, value in best_result.config.items():
    if not key.startswith("_"):
        print(f"    {key}: {value}")

In [ ]:
# Re-generate collected_results.csv (already produced automatically after sweep).
# Useful for re-running with updated results_df from section 7.
from environments.shared.scripts.sweep.results import plot_sweep_results, write_results_csv

# ── Standardized sweep CSV (same schema as Vertex AI sweep & training notebook) ──
# Convert Ray Tune results into the shared row-dict format so the
# collected_results.csv is consistent across all training workflows.
_sweep_rows = []
stage_config = STAGE_CONFIGS[STAGE]
for _, rt_row in results_df.iterrows():
    row = {"trial_id": str(rt_row.get("trial_id", "")), "stage": STAGE}
    # Hyperparameters from Ray Tune config
    for col in results_df.columns:
        if col.startswith(("ppo_", "sac_", "env_")):
            row[col] = rt_row[col]
    # Metrics
    row["best_mean_reward"] = rt_row.get("best_mean_reward")
    row["best_mean_episode_length"] = rt_row.get("best_mean_episode_length")
    row["last_mean_reward"] = rt_row.get("last_mean_reward")
    row["last_mean_episode_length"] = rt_row.get("last_mean_episode_length")
    row["mean_forward_vel"] = rt_row.get("mean_forward_vel")
    row["std_forward_vel"] = rt_row.get("std_forward_vel")
    row["mean_success_rate"] = rt_row.get("mean_success_rate")
    row["training_duration_seconds"] = rt_row.get("training_duration_seconds")
    # Curriculum thresholds from stage config
    cur = stage_config.get("curriculum_kwargs", {})
    row["reward_threshold"] = cur.get("min_avg_reward")
    row["ep_length_threshold"] = cur.get("min_avg_episode_length")
    row["forward_vel_threshold"] = cur.get("min_avg_forward_vel")
    row["success_rate_threshold"] = cur.get("min_success_rate")
    # Gate evaluation
    from environments.shared.scripts.sweep.results import _evaluate_curriculum_gate
    passed, _ = _evaluate_curriculum_gate(
        row["best_mean_reward"], row,
        row["reward_threshold"], row["ep_length_threshold"],
        row["forward_vel_threshold"], row["success_rate_threshold"],
    )
    row["stage_passed"] = passed
    _sweep_rows.append(row)

collected_csv = write_results_csv(_sweep_rows, SWEEP_DIR / "collected_results.csv")
print(f"Standardized CSV saved to: {collected_csv}")

# ── Sweep analysis plots (shared with Vertex AI sweep) ──
plot_sweep_results(collected_csv, SPECIES, ALGORITHM, save_dir=SWEEP_DIR)
print(f"Sweep analysis plots saved to: {SWEEP_DIR}")

# Also save the raw Ray Tune DataFrame for Ray-specific columns
raw_csv = SWEEP_DIR / "sweep_results_raw.csv"
results_df.to_csv(str(raw_csv), index=False)
print(f"Raw Ray Tune CSV saved to: {raw_csv}")

## 7b. Post-Sweep Deep Analysis (Standalone)

Run this section to evaluate the **top trials** with full locomotion metrics
(forward velocity, gait symmetry, cost of transport, etc.) that are **not**
captured during the Ray Tune sweep itself.

**This section is self-contained** — if your Colab session restarted after the
sweep, just re-run Sections 1–3 (Setup, Configuration, Load Species) and then
set `SWEEP_DIR` below to the Drive path from the previous sweep. You do **not**
need to re-run the Tune sweep.

In [ ]:
# ============================================================
# Standalone reload: set this to a previous sweep's Drive path
# if your Colab session restarted after the sweep finished.
# When running right after Section 6, leave as-is — SWEEP_DIR
# is already set.
# ============================================================
# SWEEP_DIR = Path("/content/drive/MyDrive/mesozoic-labs/ray_tune_sweeps/velociraptor/stage2_ppo_20250101_120000")  # uncomment & edit

TOP_K = 5  # @param {type:"integer"} — number of top trials to analyze
EVAL_EPISODES = 30  # @param {type:"integer"} — episodes per trial evaluation

print(f"Sweep directory: {SWEEP_DIR}")
print(f"Analyzing top {TOP_K} trials with {EVAL_EPISODES}-episode evaluation")

# ── Discover trial directories ─────────────────────────────────────────
trials_root = SWEEP_DIR / "trials"
assert trials_root.exists(), f"No trials directory found at {trials_root}"

# Find trial dirs that have a best_model.zip and rank by evaluations.npz
_trial_dirs = sorted(trials_root.iterdir())
_valid_trials = []
for td in _trial_dirs:
    if td.is_dir() and (td / "models" / "best_model.zip").exists():
        # Read best_mean_reward from evaluations.npz
        eval_npz = td / "evaluations.npz"
        reward = float("-inf")
        if eval_npz.exists():
            _eval_data = np.load(str(eval_npz))
            _mean_per_eval = _eval_data["results"].mean(axis=1)
            reward = float(_mean_per_eval.max())
        _valid_trials.append((td, reward))

_valid_trials.sort(key=lambda x: x[1], reverse=True)
_top_trials = _valid_trials[:TOP_K]

print(f"\nFound {len(_valid_trials)} trials with saved models, analyzing top {len(_top_trials)}:")
for i, (td, r) in enumerate(_top_trials):
    print(f"  {i+1}. {td.name}  (best_mean_reward = {r:.2f})")

In [ ]:
import logging
import pandas as pd
from stable_baselines3 import PPO, SAC

from environments.shared.evaluation import eval_policy
from environments.shared.metrics import LocomotionMetrics
from environments.shared.reporting import generate_stage_artifacts
from environments.shared.train_base import create_vec_env, _ensure_sb3

logging.basicConfig(level=logging.INFO)
_logger = logging.getLogger("post_sweep_analysis")

sb3 = _ensure_sb3()
stage_config = STAGE_CONFIGS[STAGE]
env_kwargs = stage_config["env_kwargs"].copy()
alg_cls = SAC if ALGORITHM == "sac" else PPO

# ── Evaluate each top trial ───────────────────────────────────────────
analysis_rows = []

for rank, (trial_dir, sweep_reward) in enumerate(_top_trials, 1):
    trial_name = trial_dir.name
    model_dir = trial_dir / "models"
    model_path = str(model_dir / "best_model")
    vecnorm_path = str(model_dir / "best_model_vecnorm.pkl")

    print(f"\n{'=' * 60}")
    print(f"Trial {rank}/{len(_top_trials)}: {trial_name}  (sweep reward: {sweep_reward:.2f})")
    print(f"{'=' * 60}")

    # Load model + VecNormalize
    def _make_eval_env():
        return sb3["Monitor"](SPECIES_CFG.env_class(**env_kwargs))

    eval_env = sb3["DummyVecEnv"]([_make_eval_env])
    if Path(vecnorm_path).exists():
        eval_env = sb3["VecNormalize"].load(vecnorm_path, eval_env)
        eval_env.training = False
        eval_env.norm_reward = False

    model = alg_cls.load(model_path, env=eval_env)

    # Full evaluation with LocomotionMetrics
    episode_reports = []
    for ep in range(EVAL_EPISODES):
        obs = eval_env.reset()
        metrics = LocomotionMetrics()
        total_reward = 0.0
        step = 0
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = eval_env.step(action)
            total_reward += float(rewards[0])
            step += 1
            metrics.record_step(infos[0], float(rewards[0]))
            if dones[0]:
                break
        episode_reports.append(metrics.compute())

    eval_env.close()

    agg = LocomotionMetrics.aggregate_episodes(episode_reports)

    row = {
        "rank": rank,
        "trial": trial_name,
        "sweep_reward": round(sweep_reward, 2),
        "eval_reward": round(agg.get("mean_total_reward", 0), 2),
        "eval_reward_std": round(agg.get("std_total_reward", 0), 2),
        "ep_length": round(agg.get("mean_episode_length", 0), 1),
        "fwd_vel_m/s": round(agg.get("mean_mean_forward_velocity", 0), 3),
        "fwd_vel_std": round(agg.get("std_mean_forward_velocity", 0), 3),
        "max_fwd_vel": round(agg.get("mean_max_forward_velocity", 0), 3),
        "distance_m": round(agg.get("mean_total_distance", 0), 3),
        "vel_consistency": round(agg.get("mean_velocity_consistency", 0), 3),
        "gait_symmetry": round(agg.get("mean_gait_symmetry", 0), 3),
        "stride_freq_Hz": round(agg.get("mean_stride_frequency", 0), 3),
        "cost_of_transport": round(agg.get("mean_cost_of_transport", 0), 4),
        "tilt_rad": round(agg.get("mean_mean_tilt_angle", 0), 3),
        "pelvis_height_m": round(agg.get("mean_mean_pelvis_height", 0), 3),
    }
    analysis_rows.append(row)

    _logger.info(
        "  reward=%.2f  fwd_vel=%.3f m/s  distance=%.2f m  symmetry=%.3f  CoT=%.4f",
        row["eval_reward"], row["fwd_vel_m/s"], row["distance_m"],
        row["gait_symmetry"], row["cost_of_transport"],
    )

    # Generate stage artifacts (training curves, videos)
    generate_stage_artifacts(
        SPECIES_CFG,
        stage_config,
        STAGE,
        ALGORITHM,
        stage_dir=trial_dir,
        seed=SEED,
        timesteps=TIMESTEPS_PER_TRIAL,
        record_videos=True,
        generate_graphs=True,
    )

print(f"\nPost-sweep evaluation complete for {len(analysis_rows)} trials.")

In [ ]:
from environments.shared.visualization import plot_trial_comparison

# ── Comparison table ───────────────────────────────────────────────────
analysis_df = pd.DataFrame(analysis_rows)
analysis_df = analysis_df.set_index("rank")

print(f"\n{'=' * 80}")
print(f"Post-Sweep Analysis: {SPECIES.title()} Stage {STAGE} ({ALGORITHM.upper()}) — Top {len(analysis_rows)} Trials")
print(f"{'=' * 80}\n")
display(analysis_df)

# Save to CSV
analysis_csv = SWEEP_DIR / "post_sweep_analysis.csv"
analysis_df.to_csv(str(analysis_csv))
print(f"\nAnalysis CSV saved to: {analysis_csv}")

# ── Comparison plots (shared with Vertex AI sweep) ────────────────────
comparison_path = SWEEP_DIR / "post_sweep_comparison.png"
plot_trial_comparison(
    analysis_rows,
    species=SPECIES,
    stage=STAGE,
    save_path=comparison_path,
    show=True,
)
print(f"Comparison plot saved to: {comparison_path}")

## 8. Save Results to Google Drive

Persist the sweep results CSV, best trial config, and analysis plots to Google Drive
so they survive session restarts.

In [ ]:
# Save best trial config as JSON
best_config = {k: v for k, v in best_result.config.items() if not k.startswith("_")}
best_config_with_meta = {
    "species": SPECIES,
    "algorithm": ALGORITHM,
    "stage": STAGE,
    "best_mean_reward": best_result.metrics.get("best_mean_reward"),
    "timesteps_per_trial": TIMESTEPS_PER_TRIAL,
    "num_trials": NUM_TRIALS,
    "scheduler": "ASHA",
    "hyperparameters": best_config,
}

best_config_path = SWEEP_DIR / "best_trial_config.json"
with open(str(best_config_path), "w") as f:
    json.dump(best_config_with_meta, f, indent=2, default=str)
print(f"Best trial config saved to: {best_config_path}")

# Copy best trial's model to a convenient location on Drive.
# Models may be in local trial dir or already synced to Drive.
best_trial_id = best_result.metrics.get("trial_id", "")
_local_model_dir = LOCAL_TRIALS_DIR / str(best_trial_id) / "models"
_drive_model_dir = SWEEP_DIR / "trials" / str(best_trial_id) / "models"

# Prefer local (always complete), fall back to Drive (synced copy)
best_trial_model_dir = _local_model_dir if _local_model_dir.exists() else _drive_model_dir

best_model_dest = SWEEP_DIR / "best_model"
best_model_dest.mkdir(parents=True, exist_ok=True)

import shutil

_found_any = False
for src_pattern in [f"stage{STAGE}_final.zip", f"stage{STAGE}_final_vecnorm.pkl",
                    "best_model.zip", "best_model_vecnorm.pkl"]:
    src_files = list(best_trial_model_dir.glob(src_pattern)) if best_trial_model_dir.exists() else []
    for src in src_files:
        dest = best_model_dest / src.name
        shutil.copy2(str(src), str(dest))
        print(f"  Copied: {src.name} -> {dest}")
        _found_any = True

if not _found_any:
    print(f"  Warning: No model files found in {best_trial_model_dir}")
    print(f"  Check if the best trial completed successfully.")

print(f"\nAll results saved to: {SWEEP_DIR}")
print(f"\nTo use the best model for the next stage, set LOAD_PATH to:")
print(f"  {best_model_dest / 'best_model'}")

## 9. Apply Best Hyperparameters

Use this cell to generate `--override` flags for the CLI training scripts,
so you can retrain with the best hyperparameters found by the sweep.

In [ ]:
# Generate CLI override flags for the best trial
overrides = []
for key, value in best_config.items():
    for prefix in ("ppo", "sac", "env", "curriculum"):
        if key.startswith(prefix + "_"):
            param = key[len(prefix) + 1:]
            if param == "net_arch":
                # net_arch is handled specially by the training script
                overrides.append(f"{prefix}.policy_kwargs.net_arch={value}")
            elif isinstance(value, float) and value == int(value):
                overrides.append(f"{prefix}.{param}={int(value)}")
            else:
                overrides.append(f"{prefix}.{param}={value}")
            break

override_str = " ".join(f"--override {o}" for o in overrides)

print(f"Best hyperparameters as CLI overrides:\n")
print(f"python -m environments.{SPECIES}.scripts.train_sb3 train \\")
print(f"    --stage {STAGE} \\")
print(f"    --algorithm {ALGORITHM} \\")
print(f"    --timesteps {TIMESTEPS_PER_TRIAL} \\")
for o in overrides:
    print(f"    --override {o} \\")

## 10. Cleanup

In [ ]:
ray.shutdown()
print("Ray shutdown complete.")
print(f"\nSweep results directory: {SWEEP_DIR}")
print(f"Best trial config: {best_config_path}")
print(f"Best model: {best_model_dest}")